# An Agentic Cover Letter System

A LangGraph system that turns **a job posting + your CV + your past cover letters** into a
tailored cover letter, with grounded company research, a deterministic constraint gate, a
critique/revise loop and a human approval step.

## The graph

```
START
  │
  ▼
redact ──────── one call lists the employers in your CV; Python rewrites the text once,
  │             so no later stage can leak a name it never received
  ▼
brief ───────── posting + redacted CV → ranked requirements, which evidence answers
  │             each one, the gaps, and what to lead with
  ▼
research ────── web search → facts about the company, each carrying a source URL
  │             Python discards anything it cannot attribute
  ▼
write ───────── brief + research + your voice → the letter
  │
  ▼
review ◄────┐   Python rules + one critique call → issues, three scores, edits
  │         │
  ├─ revise ┘   applies the edits
  │
  ▼
approve ─────── interrupt(): approve, or send feedback in your own words
  │
  ▼
 END           writes outputs/<date>_<company>-<role>.md
```

**7 nodes, ~5 model calls, linear.** Roughly three minutes end to end.

## Why the stages exist

A single prompt asks one model call to do four incompatible jobs at once: discover facts about
the company, choose evidence from a whole CV, obey hard constraints, and write good prose. Each
fails in its own way, and each stage below exists to catch one of them.

| Failure | Cause | Caught by |
|---|---|---|
| Wrong evidence chosen | Recency bias - the newest CV entry beats the most *relevant* one | `brief` scores evidence against weighted requirements |
| Recycled phrasing | Your own verbal tics resurface in every letter | phrases recurring across your past letters are banned outright |
| Leaked employer names | An instruction a model can silently drop under revision pressure | `redact` removes the names before any writing stage sees them |
| Unfalsifiable flattery | Nothing grounds "your mission resonates with me" | `research` keeps only claims with a visited source URL |

A note on what is deliberately *not* structured: almost everything passed between stages is
markdown, because its only consumer is the next prompt. Only three things are typed - the
employer list, the brief's company/role, and the review scores - because those are the only
three that Python itself reads.

---
## 1. Setup

```bash
uv venv --python 3.13
uv pip install langgraph langchain-anthropic python-dotenv ipykernel pypdf python-docx
```

Put your key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

Then select the `.venv` kernel.

In [ ]:
# No `from __future__ import annotations` here on purpose: IPython propagates __future__ imports
# to every later cell, which would turn the graph state's annotations into lazy strings that
# LangGraph cannot resolve when it builds its channels.
import json
import re
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import date
from pathlib import Path
from typing import Literal, TypedDict

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

import os
load_dotenv(find_dotenv(usecwd=True))
assert os.environ.get("ANTHROPIC_API_KEY"), "Put ANTHROPIC_API_KEY in a .env file next to this notebook"
print("Anthropic key loaded.")

---
## 2. Configuration

Everything the system will not decide for you.

`nameable_employers` is the anonymisation allowlist, and it defaults to naming **nothing** -
every employer becomes a generic descriptor unless you opt it in.

In [ ]:
@dataclass
class Config:
    model: str = "claude-opus-5"

    output_language: str = "English"
    min_words: int = 280
    max_words: int = 400

    nameable_employers: tuple = ()      # employers NOT listed here are replaced by descriptors
    redact_employers: bool = True
    address_hard_gaps: bool = False     # True = name unmet hard requirements in the letter

    max_revisions: int = 3
    pass_threshold: float = 4.0         # mean of the three review scores, out of 5
    web_search_max_uses: int = 6

    inputs_dir: Path = Path("inputs")
    outputs_dir: Path = Path("outputs")


cfg = Config(nameable_employers=("ASML", "Philips"))
cfg.outputs_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=2))

---
## 3. Inputs

```
inputs/
├── job_posting.txt        # or .pdf / .docx / .md
├── cv.txt                 # or .pdf / .docx
└── past_letters/          # any number of letters you wrote before
```

The past letters are never copied from. They are used to learn your voice and, more usefully, to
detect what you repeat.

In [ ]:
def read_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join((page.extract_text() or "") for page in PdfReader(str(path)).pages)
    if suffix == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(str(path)).paragraphs)
    raise ValueError(f"Unsupported file type: {path.name}")


def load_inputs(cfg: Config) -> dict:
    def find(stem):
        matches = [p for p in cfg.inputs_dir.glob(f"{stem}.*")
                   if p.suffix.lower() in {".txt", ".md", ".pdf", ".docx"}]
        if not matches:
            raise FileNotFoundError(f"No {stem}.(txt|md|pdf|docx) in {cfg.inputs_dir}/")
        return matches[0]

    letters_dir = cfg.inputs_dir / "past_letters"
    letter_paths = sorted(p for p in letters_dir.glob("*")
                          if p.suffix.lower() in {".txt", ".md", ".pdf", ".docx"}) if letters_dir.exists() else []

    data = {"job_posting": read_document(find("job_posting")),
            "cv": read_document(find("cv")),
            "past_letters": [read_document(p) for p in letter_paths]}

    print(f"posting : {len(data['job_posting'].split())} words")
    print(f"cv      : {len(data['cv'].split())} words")
    print(f"letters : {len(letter_paths)} -> {[p.name for p in letter_paths]}")
    return data


inputs = load_inputs(cfg)

---
## 4. State, schemas and helpers

The state is plain strings, lists and dicts. That is a deliberate choice: checkpointers hand
Pydantic models back as dicts after a resume, so keeping models out of the state removes a whole
category of bug at the `interrupt()` boundary.

Three schemas, and each exists because **Python** reads the result - not because the next prompt
does. Stages that only feed another prompt pass markdown.

In [ ]:
class Employer(BaseModel):
    real: str = Field(description="the organisation name exactly as written in the CV")
    generic: str = Field(description="a descriptor precise about industry, region and scale but "
                                     "never identifying, e.g. 'a Dutch telecommunications operator'")


class EmployerList(BaseModel):
    employers: list[Employer]


class Brief(BaseModel):
    company: str
    role: str
    markdown: str = Field(description="the full brief in markdown")


class Review(BaseModel):
    company_specificity: int = Field(ge=1, le=5)
    evidence_concreteness: int = Field(ge=1, le=5)
    requirement_coverage: int = Field(ge=1, le=5)
    edits: list[str] = Field(description="specific, applicable edits in priority order")


class State(TypedDict, total=False):
    job_posting: str
    cv: str
    past_letters: list[str]

    redactions: dict         # real name -> generic descriptor
    cv_clean: str
    company: str
    role: str
    brief: str
    research: str
    sources: list

    draft: str
    hard_issues: list
    soft_issues: list
    scores: dict
    score: float
    edits: list
    revision: int
    human_feedback: str
    final_letter: str


def model(max_tokens: int = 16000) -> ChatAnthropic:
    return ChatAnthropic(model=cfg.model, max_tokens=max_tokens)


def ask(schema, system: str, user: str, attempts: int = 3):
    """A structured call, retried on a malformed response.

    Keep max_tokens generous: Claude Opus 5 thinks by default and those tokens come out of the
    same budget, so starving it truncates the tool call rather than the prose."""
    chain = model().with_structured_output(schema)
    messages = [SystemMessage(content=system), HumanMessage(content=user)]
    for attempt in range(1, attempts + 1):
        try:
            return chain.invoke(messages)
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f"    ({schema.__name__} attempt {attempt} failed: {type(exc).__name__}, retrying)")


def prose(system: str, user: str, max_tokens: int = 12000) -> str:
    """A free-text call, guarding against an empty completion."""
    message = model(max_tokens).invoke([SystemMessage(content=system), HumanMessage(content=user)])
    text = message.text if isinstance(message.text, str) else message.text()
    if not text.strip():
        raise RuntimeError("Model returned no text - thinking consumed the budget; raise max_tokens.")
    return text.strip()


def banned_phrases(letters: list) -> list:
    """Any 4-7 word phrase appearing in two or more of your past letters is a tic, not a choice.

    Pure Python - no model call. This is why voice analysis does not need a node of its own."""
    if len(letters) < 2:
        return []

    def shingles(text, n):
        words = re.findall(r"[a-z']+", text.lower())
        return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}

    counts = Counter()
    for letter in letters:
        seen = set()
        for n in range(4, 8):
            seen |= shingles(letter, n)
        counts.update(seen)

    repeated = {p for p, c in counts.items() if c >= 2}
    maximal = [p for p in repeated if not any(p != q and p in q for q in repeated)]
    return sorted(maximal, key=len, reverse=True)[:30]


print("state, schemas and helpers defined")

---
## 5. Node: `redact`

One call lists the employers in your CV. Python decides which may be named, rewrites the CV text
once, and every later stage receives only the rewritten version.

The point is that redaction becomes a property of the data rather than an instruction anyone has
to remember. A name that was never in the context cannot leak from it - and because the
substitution happens in one place, verifying it is a single regex rather than an audit of every
prompt.

In [ ]:
EMPLOYER_SYSTEM = """List every organisation named in this CV as an employer, client, or project
host - including universities the person studied at.

For each, give the name exactly as written, plus a generic descriptor that captures industry,
region and scale precisely enough to be meaningful in a cover letter but never identifies the
organisation. Good: "a Dutch telecommunications operator", "a European semiconductor equipment
manufacturer". Bad: "a company", "a well-known tech firm"."""


def redact(state: State) -> dict:
    if not cfg.redact_employers:
        return {"cv_clean": state["cv"], "redactions": {}}

    found = ask(EmployerList, EMPLOYER_SYSTEM, f"CV:\n\n{state['cv']}")
    allow = {name.strip().lower() for name in cfg.nameable_employers}
    mapping = {e.real: e.generic for e in found.employers if e.real.strip().lower() not in allow}

    # Longest first, so "Bridgestone Mobility Solutions" is replaced before "Bridgestone".
    def scrub(text: str) -> str:
        for real in sorted(mapping, key=len, reverse=True):
            text = re.sub(rf"\b{re.escape(real)}\b", mapping[real], text, flags=re.IGNORECASE)
        return text

    print(f"[redact] {len(found.employers)} organisations found, {len(mapping)} redacted, "
          f"{len(found.employers) - len(mapping)} nameable")
    return {
        "cv_clean": scrub(state["cv"]),
        "past_letters": [scrub(letter) for letter in state.get("past_letters", [])],
        "redactions": mapping,
    }

---
## 6. Node: `brief`

One call turns the posting and your CV into a working brief: what the job actually is, the
requirements ranked by weight, which of your evidence answers each one, where the gaps are, and
what the letter should lead with.

This is the node that decides what the letter argues. Asked simply to "write a letter", a model
reaches for whatever is most recent or most impressive-sounding. Asked to match weighted
requirements against specific evidence, it has to justify the choice - which is how an older,
plainer project wins when it is the better fit.

In [ ]:
BRIEF_SYSTEM = """You brief a writer who will draft a cover letter. You do not write the letter.

Produce markdown with these sections:

## The job
One plain sentence: what this person does all day. Strip the marketing. Write it in English even
if the posting is in another language.

## Requirements
Every stated requirement, weighted 1-5 by how central it is to the role, marked must-have or
nice-to-have, and marked HARD FILTER where failing it alone disqualifies (residence, language,
degree, work authorisation). Weight by what the responsibilities emphasise, not by bullet order.
Ignore the benefits section entirely.

## Evidence
For each requirement above weight 3, name the specific CV item that answers it and say in one
line why it does. Mark honestly as strong, partial, or nothing. A generous match here produces a
letter that collapses at interview.

## Gaps
Requirements with no evidence behind them, stated plainly. {gap_policy}

## Lead with
The two or three items that should carry the letter, ordered, with one line each on the argument
they make. Choose by weighted relevance, never by recency and never by how impressive an item
sounds on its own.

Refer to organisations exactly as the CV gives them. Several are described generically on
purpose - never substitute a real name or speculate about one."""


def brief(state: State) -> dict:
    policy = ("The letter WILL address these: plan one honest sentence for each, framed around "
              "what is being done about it."
              if cfg.address_hard_gaps else
              "The letter will NOT mention these. Note them here for the writer's awareness only, "
              "so nothing in the letter accidentally draws attention to them.")

    result = ask(Brief, BRIEF_SYSTEM.format(gap_policy=policy),
                 f"JOB POSTING:\n\n{state['job_posting']}\n\n\nCV:\n\n{state['cv_clean']}")

    print(f"[brief] {result.company} - {result.role} ({len(result.markdown.split())} words)")
    return {"company": result.company, "role": result.role, "brief": result.markdown,
            "revision": 0}

---
## 7. Node: `research`

Two calls. The first searches; the second turns the notes into facts that each carry a URL. Then
Python drops any fact whose URL was never actually visited.

That filter is the whole point. A model asked to cite its sources will occasionally attach a
plausible URL to a half-remembered fact, and the opening paragraph is exactly where that does the
most damage. A claim that cannot be sourced does not survive to the draft.

In [ ]:
SEARCH_PROMPT = """Research {company} for someone applying to be a {role}.

Find specific, checkable facts - not marketing adjectives:
1. What the company does, its scale and market position.
2. How the part of the business this role sits in actually works day to day.
3. Concrete recent developments - expansion, technology investment, published engineering work.
   Prefer the last two years.
4. Documented engineering or data practice: tech blogs, conference talks, reports.
5. Stated culture and working practices, in the company's own words where possible.

Report what you find with sources. Say so when something is not findable rather than guessing."""

GROUND_SYSTEM = """Turn research notes into a markdown list of facts about the company.

Every line must be a specific, checkable fact followed by the URL it came from, in this exact
form:

- fact goes here [https://source.url]

Use only URLs from the allowed list. Discard anything you cannot attribute to one of them,
including things you happen to know independently. A fact is never an adjective: "operates its
own delivery fleet and installs appliances in the home" is a fact; "is an innovative company" is
not.

After the facts, add a `## Angles` section: two or three openings the letter could take, each one
line, ranked by how specifically it ties to this company. An angle any applicant to any company
could write is worthless - say so and discard it."""


def research(state: State) -> dict:
    searcher = model().bind_tools(
        [{"type": "web_search_20260209", "name": "web_search", "max_uses": cfg.web_search_max_uses}])
    result = searcher.invoke([HumanMessage(
        content=SEARCH_PROMPT.format(company=state["company"], role=state["role"]))])

    sources, seen = [], set()
    for block in result.content:
        if not isinstance(block, dict) or block.get("type") != "web_search_tool_result":
            continue
        content = block.get("content")
        if not isinstance(content, list):          # an error object rather than results
            print(f"[research] search error: {content}")
            continue
        for item in content:
            if item.get("url") and item["url"] not in seen:
                seen.add(item["url"])
                sources.append({"title": item.get("title", ""), "url": item["url"]})

    notes = result.text if isinstance(result.text, str) else result.text()
    allowed = "\n".join(f"- {s['url']} ({s['title']})" for s in sources) or "(none)"
    grounded = prose(GROUND_SYSTEM,
                     f"COMPANY: {state['company']}\nROLE: {state['role']}\n\n"
                     f"NOTES:\n{notes}\n\nALLOWED SOURCE URLS:\n{allowed}")

    # keep only lines whose cited URL was actually visited
    kept, dropped = [], 0
    for line in grounded.splitlines():
        cited = re.findall(r"https?://[^\s\]]+", line)
        if cited and not any(url in seen for url in cited):
            dropped += 1
            continue
        kept.append(line)

    print(f"[research] {len(sources)} sources visited, {dropped} ungrounded claim(s) dropped")
    return {"research": "\n".join(kept), "sources": sources}

---
## 8. Node: `write`

Drafts from the brief and the researched facts. It never sees your raw CV - only the brief's
selection of it - which is what stops the letter from turning into a keyword dump of everything
you have ever done.

In [ ]:
WRITE_SYSTEM = """You write a cover letter from a brief. Follow the brief's argument and its
choice of evidence.

Structure: an opening paragraph on why this company specifically, two or three body paragraphs on
fit, and a short closing.

Hard rules:
- Language: {language}. Length {min_words}-{max_words} words.
- State no fact about the hiring company that is not in the researched facts.
- Invent no metric, date or responsibility that is not in the brief.
- Refer to organisations exactly as the brief names them. Several are described generically on
  purpose; never substitute a real name or guess at one.
- Do not use these phrases. They are the applicant's own, recycled from their past letters:
{banned}

Style:
- Open with substance. Never "I am excited to apply for" or "I am writing to apply for".
- Concrete nouns and verbs. If a sentence would survive swapping in a different company or a
  different applicant, it is filler - rewrite it.
- Every claim carries its evidence in the same sentence. No adjective stands alone as a
  qualification.
- The closing is short and forward-looking, and does not summarise what you just said.

Output the letter only, salutation through sign-off. No preamble, no commentary."""


def write_system(state: State) -> str:
    banned = banned_phrases(state.get("past_letters", []))
    return WRITE_SYSTEM.format(
        language=cfg.output_language, min_words=cfg.min_words, max_words=cfg.max_words,
        banned="\n".join(f'  - "{p}"' for p in banned) or "  (none)")


def write(state: State) -> dict:
    letters = state.get("past_letters", [])
    voice = "\n\n--- past letter ---\n\n".join(letters[:3]) or "(none supplied)"

    text = prose(write_system(state),
                 f"BRIEF:\n{state['brief']}\n\n"
                 f"RESEARCHED FACTS ABOUT {state['company'].upper()} "
                 f"(the only company facts you may state):\n{state['research']}\n\n"
                 f"THE APPLICANT'S PAST LETTERS, for voice only - do not reuse their content:\n{voice}")

    print(f"[write] {len(text.split())} words")
    return {"draft": text}

---
## 9. Node: `review`

Two halves in one node, deliberately.

**Rules, in Python.** Length, structure, placeholders, recycled phrasing, and whether a redacted
employer name appears. Asking a model whether it obeyed its own instructions is not verification;
these are facts about a string, so a string checks them.

**Judgement, in one call.** Three scores and a list of applicable edits. Each edit must name what
to change and what to change it to - "be more specific" is not an edit. The mean is computed in
Python, because a model asked for an overall score tends to rationalise a number it chose first.

In [ ]:
GENERIC_OPENERS = ["i am excited to apply", "i am writing to apply", "i would like to apply",
                   "i am thrilled to apply", "i am writing to express my interest"]
PLACEHOLDER_RE = re.compile(r"(\[[A-Za-z][^\]]{0,40}\]|\{\{.*?\}\}|\bTODO\b|\bXXXX?\b)")

REVIEW_SYSTEM = """You are a hiring-side reviewer scoring a cover letter. Score each dimension
1-5 independently; do not compute an overall score.

- company_specificity: could this be sent to a competitor after a find-replace? If yes, 1. A 5
  requires the opening to rest on facts true of this company and no other.
- evidence_concreteness: named actions, methods and results, against self-describing adjectives.
  "Strong Python skills" is a 1. A named method with a stated outcome is a 5.
- requirement_coverage: are the heaviest must-haves in the brief addressed with evidence? An
  unaddressed weight-5 must-have caps this at 2.

`edits`: the changes that would raise the score most, in priority order, each naming what to
change and what to change it to. Include fixes for anything the automated checks flagged. If a
sentence is filler, quote it and say what replaces it."""


def review(state: State) -> dict:
    text, lowered = state["draft"], state["draft"].lower()
    hard, soft = [], []

    for real, generic in state.get("redactions", {}).items():
        if re.search(rf"\b{re.escape(real)}\b", text, flags=re.IGNORECASE):
            hard.append(f"redaction: '{real}' must not be named - use '{generic}'")

    words = len(text.split())
    if words > cfg.max_words:
        hard.append(f"length: {words} words, limit {cfg.max_words} - cut {words - cfg.max_words}")
    elif words < cfg.min_words:
        hard.append(f"length: {words} words, minimum {cfg.min_words}")

    for match in set(PLACEHOLDER_RE.findall(text)):
        hard.append(f"placeholder: unfilled {match!r}")

    if len([p for p in re.split(r"\n\s*\n", text) if p.strip()]) < 3:
        hard.append("structure: fewer than three paragraphs")

    for phrase in banned_phrases(state.get("past_letters", [])):
        if len(phrase) > 15 and phrase in lowered:
            soft.append(f'recycled: "{phrase}" appears in a past letter')
    for opener in GENERIC_OPENERS:
        if opener in lowered[:400]:
            soft.append(f'generic opener: "{opener}"')

    flagged = "\n".join(f"- {i}" for i in hard + soft) or "(none)"
    result = ask(Review, REVIEW_SYSTEM,
                 f"BRIEF THE LETTER SHOULD EXECUTE:\n{state['brief']}\n\n"
                 f"RESEARCHED FACTS AVAILABLE:\n{state['research']}\n\n"
                 f"AUTOMATED CHECKS FLAGGED:\n{flagged}\n\n"
                 f"THE LETTER:\n---\n{state['draft']}\n---")

    scores = {"company_specificity": result.company_specificity,
              "evidence_concreteness": result.evidence_concreteness,
              "requirement_coverage": result.requirement_coverage}
    mean = round(sum(scores.values()) / len(scores), 2)

    print(f"[review] {mean}/5  " + "  ".join(f"{k.split('_')[0]}={v}" for k, v in scores.items())
          + f"  |  {len(hard)} hard, {len(soft)} soft")
    for issue in (hard + soft)[:5]:
        print(f"    {issue}")

    return {"hard_issues": hard, "soft_issues": soft, "scores": scores,
            "score": mean, "edits": result.edits}

---
## 10. `revise`, the router, and `approve`

The loop stops on the first of: nothing left to fix, or the revision budget spent. `revise` edits
rather than rewrites, so paragraphs that already work survive the loop.

`approve` suspends the graph with `interrupt()`. Resume with either:

```python
graph.invoke(Command(resume={"action": "approve"}), config)
graph.invoke(Command(resume={"action": "revise", "feedback": "..."}), config)
```

Your feedback outranks the reviewer and resets the revision budget - it is your letter.

In [ ]:
REVISE_SYSTEM = """You revise a cover letter by applying specific edits.

Change what the edits identify and leave the rest alone. This is editing, not rewriting: a
paragraph nobody complained about should come back recognisably intact. Every hard issue must be
resolved. All the original drafting rules still apply."""


def revise(state: State) -> dict:
    issues = "\n".join(f"- {i}" for i in state["hard_issues"] + state["soft_issues"]) or "(none)"
    edits = "\n".join(f"- {e}" for e in state["edits"]) or "(none)"
    feedback = state.get("human_feedback", "")

    text = prose(REVISE_SYSTEM + "\n\n" + write_system(state),
                 f"CURRENT LETTER:\n---\n{state['draft']}\n---\n\n"
                 f"ISSUES THAT MUST BE FIXED:\n{issues}\n\n"
                 f"REVIEWER EDITS:\n{edits}\n\n"
                 + (f"THE APPLICANT'S OWN FEEDBACK - this outranks everything above:\n{feedback}\n\n"
                    if feedback else "")
                 + f"BRIEF:\n{state['brief']}\n\nRESEARCHED FACTS:\n{state['research']}")

    revision = state.get("revision", 0) + 1
    print(f"[revise] pass {revision} -> {len(text.split())} words")
    return {"draft": text, "revision": revision, "human_feedback": ""}


def route(state: State) -> Literal["revise", "approve"]:
    revision = state.get("revision", 0)
    needs_work = bool(state["hard_issues"]) or state["score"] < cfg.pass_threshold

    if not needs_work:
        print(f"[route] passed ({state['score']}/5) -> approve")
        return "approve"
    if revision >= cfg.max_revisions:
        print(f"[route] revision budget spent ({revision}/{cfg.max_revisions}) -> approve")
        return "approve"

    reason = "hard issues" if state["hard_issues"] else f"score {state['score']} < {cfg.pass_threshold}"
    print(f"[route] {reason} -> revise (pass {revision + 1}/{cfg.max_revisions})")
    return "revise"


def approve(state: State) -> Command:
    decision = interrupt({
        "draft": state["draft"],
        "words": len(state["draft"].split()),
        "score": state["score"],
        "scores": state["scores"],
        "issues": state["hard_issues"] + state["soft_issues"],
        "edits": state["edits"],
        "sources": state.get("sources", []),
        "resume_with": 'Command(resume={"action": "approve"}) or '
                       'Command(resume={"action": "revise", "feedback": "..."})',
    })

    if isinstance(decision, str):
        decision = ({"action": "approve"} if decision.strip().lower() in {"approve", "ok", "yes", "y"}
                    else {"action": "revise", "feedback": decision})

    if decision.get("action") != "approve":
        print("[approve] revision requested by the applicant")
        return Command(goto="revise",
                       update={"human_feedback": decision.get("feedback", ""), "revision": 0})

    slug = re.sub(r"[^a-z0-9]+", "-", f"{state['company']}-{state['role']}".lower()).strip("-")[:60]
    path = cfg.outputs_dir / f"{date.today().isoformat()}_{slug}.md"
    path.write_text(state["draft"], encoding="utf-8")
    print(f"[approve] letter -> {path}")
    return Command(goto=END, update={"final_letter": state["draft"]})

---
## 11. Assemble the graph

Linear, with one loop. `approve` returns a `Command`, so it chooses its own next step rather than
needing conditional edges.

In [ ]:
builder = StateGraph(State)

for name, fn in [("redact", redact), ("brief", brief), ("research", research),
                 ("write", write), ("review", review), ("revise", revise), ("approve", approve)]:
    builder.add_node(name, fn)

builder.add_edge(START, "redact")
builder.add_edge("redact", "brief")
builder.add_edge("brief", "research")
builder.add_edge("research", "write")
builder.add_edge("write", "review")
builder.add_conditional_edges("review", route, {"revise": "revise", "approve": "approve"})
builder.add_edge("revise", "review")

graph = builder.compile(checkpointer=InMemorySaver())
print("graph compiled")

In [ ]:
from IPython.display import Image

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:                     # the PNG comes from a remote renderer
    print(f"(diagram unavailable: {exc})\n")
    print(graph.get_graph().draw_mermaid())

---
## 12. Run it

Runs until `approve`, then suspends. Around three minutes and five model calls.

In [ ]:
config = {"configurable": {"thread_id": "coolblue-ds-delivery"}, "recursion_limit": 40}

result = graph.invoke(inputs, config)
print("\n" + "=" * 70)
print("PAUSED FOR REVIEW" if "__interrupt__" in result else "FINISHED")

In [ ]:
payload = result["__interrupt__"][0].value

print(f"SCORE {payload['score']}/5   {payload['words']} words\n")
for name, value in payload["scores"].items():
    print(f"  {value}/5  {name}")

if payload["issues"]:
    print("\nOUTSTANDING ISSUES")
    for issue in payload["issues"]:
        print("  - " + issue)

print("\nEDITS STILL ON THE TABLE")
for edit in payload["edits"][:5]:
    print("  - " + edit)

print(f"\nSOURCES ({len(payload['sources'])})")
for source in payload["sources"][:8]:
    print(f"  - {source['title'][:60]}  {source['url']}")

display(Markdown("---\n### Draft\n\n" + payload["draft"]))

### Approve, or send it back

```python
# send it back instead:
result = graph.invoke(
    Command(resume={"action": "revise",
                    "feedback": "Cut the third paragraph. Lead with the forecasting work."}),
    config,
)
```

In [ ]:
result = graph.invoke(Command(resume={"action": "approve"}), config)
display(Markdown("## Final letter\n\n" + result["final_letter"]))

---
## 13. The next application

```python
new_inputs = {
    "job_posting": Path("inputs/other_posting.txt").read_text(),
    "cv": inputs["cv"],
    "past_letters": inputs["past_letters"],
}
graph.invoke(new_inputs, {"configurable": {"thread_id": "other-role"}, "recursion_limit": 40})
```

Use a fresh `thread_id` per application, or the checkpointer will resume the previous run.

## Worth changing

**Cost.** Every node uses `cfg.model`. Pointing `review` and `redact` at `claude-sonnet-5` would
cut the bill meaningfully; test the reviewer first, since a weaker judge scores generously.

**Calibrate the reviewer.** The three dimensions encode one opinion of a good letter. Score a few
of your own by hand, compare, and adjust the descriptions until they agree.

**Persistence.** Swap `InMemorySaver` for `SqliteSaver` and a paused review survives a kernel
restart.

**Cache the redaction.** `redact` re-runs on every application even though your CV rarely changes.
Keying its result on a hash of the CV text would save a call per run.

**A groundedness check with teeth.** The URL filter catches a fabricated *source*; it does not
catch a real source being misread. Sentence-level entailment against the retrieved pages would.
That is the one check this system does not have.